# Week 7c: Transformer vs scikit-learn

**The question:** your Week 2 categorizer is 96% accurate. Would a fine-tuned transformer be better, and is it worth it?

Three contenders, trained on exactly the same rows:
1. `tfidf_logreg`: your Week 2 model (character n-grams + logistic regression)
2. `minilm_logreg`: a pretrained sentence model turns each description into 384 numbers (no fine-tuning), then logistic regression
3. `distilbert`: a pretrained transformer, **fine-tuned** on your data

Three test sets: **seen** merchants (the usual random split), **unseen** merchants (held out completely), and **new** real Philippine merchants.

Everything here ends up in `blog/results.json` and `blog/figures/`, and your blog post quotes it. Spec: `docs/superpowers/specs/2026-10-23-deep-learning-design.md`

## Setup (given)

In [ ]:
import importlib.util
import json
import statistics
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

sys.path.insert(0, '../service')
from clean import clean_description

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELS = ['tfidf_logreg', 'minilm_logreg', 'distilbert']

df = pd.read_csv('../data/fake_transactions.csv')
df['Clean'] = df['Description'].fillna('').str.strip().apply(clean_description)
data = df[df['Clean'] != ''].reset_index(drop=True)

# One label order for everything (DistilBERT predicts numbers; these turn them back into names)
LABELS = sorted(data['Category'].unique())
LABEL2ID = {c: i for i, c in enumerate(LABELS)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

new = pd.read_csv('../datasets/new_merchants.csv').rename(columns={'category': 'Category'})
new['Clean'] = new['description'].apply(clean_description)
assert set(new['Category']) <= set(LABELS), set(new['Category']) - set(LABELS)
print(len(data), 'fake rows,', len(new), 'new merchants,', len(LABELS), 'categories | device:', DEVICE)

### Merchant key (given)
Each fake description comes from a merchant in the generator (`JOLLIBEE*4821`, `jollibee makati` → **jollibee**). Matching the generator's merchant names inside the cleaned text recovers it, so the unseen split can hold out *every spelling* of a merchant.

In [ ]:
spec = importlib.util.spec_from_file_location('gen', '../scripts/generate_fake_data.py')
gen = importlib.util.module_from_spec(spec); spec.loader.exec_module(gen)
NAMES = [m for merchants, *_ in gen.EXPENSES.values() for m in merchants] + [b[0] for b in gen.BILLS] \
        + ['Payroll', 'Upwork', 'Client payment', 'Fiverr', 'Gift', 'Refund', 'Sold item']
squash = lambda text: clean_description(text).replace(' ', '')   # 'MAX'SRESTAURANT*12' and "Max's Restaurant" match
KEYS = sorted({squash(n) for n in NAMES}, key=len, reverse=True)   # longest first: 'grabfood' before 'grab'

def merchant_of(text):
    s = squash(text)
    return next((k for k in KEYS if k in s), None)

data['Merchant'] = data['Description'].apply(merchant_of)
for d in new['description']:
    assert merchant_of(d) is None, f'{d!r} is not new: it contains a fake-data merchant'
print(data['Merchant'].isna().sum(), 'rows without a merchant;', data['Merchant'].nunique(), 'merchants')

### The three test sets (given)

In [ ]:
seen_train, seen_test = train_test_split(data, test_size=0.2, stratify=data['Category'], random_state=42)

grouped = data.dropna(subset=['Merchant'])
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(grouped, groups=grouped['Merchant']))
unseen_train, unseen_test = grouped.iloc[train_idx], grouped.iloc[test_idx]
assert not set(unseen_train['Merchant']) & set(unseen_test['Merchant']), 'a merchant is on both sides'

print(f'seen:   {len(seen_train)} train / {len(seen_test)} test')
print(f'unseen: {len(unseen_train)} train / {len(unseen_test)} test, held-out merchants: {sorted(unseen_test["Merchant"].unique())}')
print('categories never seen in unseen training:', set(LABELS) - set(unseen_train['Category']))

**Think:** look at the held-out merchants and the last line. Is any category missing from the unseen training data? What can **no** model do then, however clever?

### Helpers (given)

In [ ]:
def score(y_true, y_pred):
    assert set(y_pred) <= set(LABELS), 'predictions must be category names'
    return {'accuracy': round(accuracy_score(y_true, y_pred), 4),
            'macro_f1': round(f1_score(y_true, y_pred, average='macro', labels=sorted(set(y_true)), zero_division=0), 4)}

def latency_ms(predict_one, texts, warmup=20, runs=200):
    """Median milliseconds per single prediction, on the CPU with one thread, the same for every model."""
    threads = torch.get_num_threads(); torch.set_num_threads(1)
    try:
        for t in texts[:warmup]:
            predict_one(t)
        times = []
        for i in range(runs):
            start = time.perf_counter(); predict_one(texts[i % len(texts)]); times.append(time.perf_counter() - start)
        return round(statistics.median(times) * 1000, 3)
    finally:
        torch.set_num_threads(threads)

def size_mb(path):
    p = Path(path)
    files = [p] if p.is_file() else [f for f in p.rglob('*') if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1e6, 2)

sample_texts = list(seen_test['Clean'])

---
## 1. The three models
Each `fit_…(texts, labels)` returns `(model, predict)`, where `predict(texts)` returns a **list of category names**.

**`fit_tfidf`:** your Week 2 pipeline: `make_pipeline(TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)), LogisticRegression(max_iter=1000))`.

In [ ]:
# TODO


**`fit_minilm`:** load `encoder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)` **once** (outside the function). `encoder.encode(texts)` gives a 384-number vector per description; fit `LogisticRegression(max_iter=2000)` on those. The encoder is never trained here: it only *reads*.

In [ ]:
# TODO


**`fit_distilbert(texts, labels, out_dir)`:** fine-tuning.
- `tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')` (once)
- an `encode(texts, labels)` helper that tokenizes with `truncation=True, max_length=32, padding='max_length'` and returns a list of dicts, adding `'labels': LABEL2ID[label]`
- `AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)`
- `TrainingArguments(output_dir=out_dir, num_train_epochs=3, learning_rate=5e-5, per_device_train_batch_size=32, fp16=torch.cuda.is_available(), seed=42, save_strategy='no', report_to='none')`, then `Trainer(...).train()`
- `predict` tokenizes a batch (`return_tensors='pt'`, moved to `model.device`), takes `logits.argmax(-1)` and maps through `ID2LABEL`. Wrap it in `torch.no_grad()`.

**Think:** the new top layer (11 outputs) starts random, but every other layer starts from what DistilBERT learned reading Wikipedia and books. Why might that help with merchants it has never seen?

In [ ]:
# TODO


---
## 2. Train and score
For each experiment, train all three models on its training part and score them:
- **seen**: train on `seen_train`, score on `seen_test`
- **unseen**: train on `unseen_train`, score on `unseen_test` **and** on `new` (those models have never seen any of those merchants)

Keep `predictions[(model, test_set)]`, `fitted[(model, experiment)] = (obj, predict)` and the training time of each model in the **seen** experiment as `train_s[model]` (seconds). Print `score(...)` as you go. DistilBERT takes a few minutes per experiment on your GPU.

In [ ]:
# TODO


---
## 3. Size and speed
**Size:** save each seen-experiment model under `../models/` (`joblib.dump` for scikit-learn; `encoder.save(...)` and `model.save_pretrained(...)` for the others) and measure it with `size_mb`. MiniLM's size is the encoder folder **plus** its logistic regression. Save a dict `sizes`.

In [ ]:
# TODO


**Speed:** a server categorizes **one** description per request, usually on a CPU. Time each model with `latency_ms(predict_one, sample_texts)`: load CPU copies (`SentenceTransformer(path, device='cpu')`, `AutoModelForSequenceClassification.from_pretrained(path)`), and write a `predict_one(text)` for each. Save a dict `latency`.

In [ ]:
# TODO


---
## 4. Where do they fail?
Show the **new** merchants that at least one model got wrong, with each model's prediction side by side.

In [ ]:
# TODO


**Think:** which mistakes are *reasonable* (a human might hesitate too), and which show a model only matching letters? Which model would you trust with a merchant it has never seen?

**Your answer:** TODO

---
## 5. Export (given)
Writes the numbers and charts your blog post uses.

In [ ]:
results = {
    'models': {m: {'size_mb': sizes[m], 'latency_ms': latency[m], 'train_s': train_s[m]} for m in MODELS},
    'scores': {m: {t: score({'seen': seen_test, 'unseen': unseen_test, 'new': new}[t]['Category'], predictions[(m, t)])
                   for t in ['seen', 'unseen', 'new']} for m in MODELS},
    'rows': {'seen_train': len(seen_train), 'seen_test': len(seen_test), 'unseen_train': len(unseen_train),
             'unseen_test': len(unseen_test), 'new': len(new)},
}
Path('../blog/figures').mkdir(parents=True, exist_ok=True)
Path('../blog/results.json').write_text(json.dumps(results, indent=1))

acc = pd.DataFrame({m: {t: results['scores'][m][t]['accuracy'] for t in ['seen', 'unseen', 'new']} for m in MODELS})
ax = acc.plot.bar(rot=0, ylim=(0, 1), figsize=(7, 4), title='Accuracy by test set')
ax.set_ylabel('accuracy'); plt.tight_layout(); plt.savefig('../blog/figures/accuracy.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for m in MODELS:
    ax.scatter(sizes[m], latency[m], s=80); ax.annotate(m, (sizes[m], latency[m]), xytext=(6, 4), textcoords='offset points')
ax.set(xscale='log', yscale='log', xlabel='model size (MB)', ylabel='CPU ms per prediction', title='Size vs speed')
plt.tight_layout(); plt.savefig('../blog/figures/speed-size.png', dpi=150); plt.show()

assert all(results['scores'][m][t] for m in MODELS for t in ['seen', 'unseen', 'new'])
print(json.dumps(results['scores'], indent=1))

**Is it worth it?** For Spendify (one user, a free Vercel server, a suggestion chip that must feel instant), would you switch to the transformer? Use your numbers.

**Your answer:** TODO